In [1]:
import os
from pathlib import Path 
import yaml 


In [2]:
os.chdir("..")

In [3]:
print(os.getcwd())

e:\project_archive\new project


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

import optuna 
import wandb 

e:\project_archive\new project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# import parameters 

model_params = Path("config/model_params.yaml")

with open(model_params, "r") as f:
    config = yaml.safe_load(f)

display(config)

{'RandomForest': {'n_estimators': {'min': 100, 'max': 400},
  'max_depth': {'min': 5, 'max': 20},
  'min_samples_split': {'min': 2, 'max': 10},
  'min_samples_leaf': {'min': 1, 'max': 5},
  'max_features': ['sqrt', 'log2'],
  'criterion': ['gini', 'entropy'],
  'bootstrap': [True],
  'class_weight': ['balanced', 'balanced_subsample']}}

In [6]:
config["RandomForest"]["n_estimators"]["min"]

100

In [7]:
# load X_train and y_train
import pandas as pd 
import numpy as np

X_train_path = Path("data/processed/X_train.csv")
y_train_path = Path("data/processed/y_train.npy")

X_train = pd.read_csv(X_train_path)
y_train = np.load(y_train_path)

print(f"X_train loaded successfully: {X_train.shape}")
print(f"y_train loaded successfully: {len(y_train)}")

X_train loaded successfully: (3096, 36)
y_train loaded successfully: 3096


In [8]:
X_test_path = Path("data/processed/X_test.csv")
y_test_path = Path("data/processed/y_test.npy")

X_test = pd.read_csv(X_test_path)
print("X_test loaded successfully.")

y_test = np.load(y_test_path)
print("y_test loaded successfully")

X_test loaded successfully.
y_test loaded successfully


In [9]:
def suggest_params(trial, config):
    return {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            config["RandomForest"]["n_estimators"]["min"],
            config["RandomForest"]["n_estimators"]["max"],
            step=50,
        ),
        "criterion": trial.suggest_categorical(
            "criterion",
            config["RandomForest"]["criterion"],
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            config["RandomForest"]["max_depth"]["min"],
            config["RandomForest"]["max_depth"]["max"],
        ),
        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            config["RandomForest"]["min_samples_split"]["min"],
            config["RandomForest"]["min_samples_split"]["max"],
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            config["RandomForest"]["min_samples_leaf"]["min"],
            config["RandomForest"]["min_samples_leaf"]["max"],
        ),
        "max_features": trial.suggest_categorical(
            "max_features",
            config["RandomForest"]["max_features"],
        ),
        "bootstrap": trial.suggest_categorical(
            "bootstrap",
            config["RandomForest"]["bootstrap"],
        ),
        "class_weight": trial.suggest_categorical(
            "class_weight",
            config["RandomForest"]["class_weight"],
        ),
        "random_state": 42,
        "n_jobs": -1,
    }

In [14]:
import optuna
import wandb

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier


def objective(trial, X, y, config, run):

    params = suggest_params(
        trial=trial,
        config=config
    )

    model = RandomForestClassifier(
        **params
        )

    cv_strategy = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
        estimator=model,
        X=X,
        y=y,
        cv=cv_strategy,
        scoring="f1_macro",
        n_jobs=-1,
    )

    score = scores.mean()

    run.log(
        {
            "trial": trial.number,
            "f1_macro_avg": score,
        }
    )

    return score

In [15]:
# train.py
from sklearn.metrics import f1_score

with wandb.init(
        project="student-drop-enroll-grad-preds",
        tags=["RandomForest", "Optuna"]
    ) as run:
     
      study = optuna.create_study(
            direction='maximize'
      )

      study.optimize(
            lambda trial: objective(
                  trial,
                  X_train,
                  y_train,
                  config=config,
                  run=run
            ),
            n_trials=50
      )

      best_params = study.best_params

      model = RandomForestClassifier(
            **best_params
      )

      model.fit(X_train, y_train)

      pred = model.predict(X_test)

      f1 = f1_score(
            y_test,
            pred,
            average='macro'
      )

      run.summary["best_cv_score"] = study.best_value
      run.summary["test_f1_macro"] = f1
      run.summary["best_params"] = best_params





[I 2026-06-25 23:26:56,122] A new study created in memory with name: no-name-90223708-6310-4a64-986f-e7f266bc6889
[I 2026-06-25 23:27:02,420] Trial 0 finished with value: 0.7061011234513103 and parameters: {'n_estimators': 250, 'criterion': 'gini', 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'bootstrap': True, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7061011234513103.
[I 2026-06-25 23:27:05,811] Trial 1 finished with value: 0.71083613895946 and parameters: {'n_estimators': 100, 'criterion': 'gini', 'max_depth': 13, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': True, 'class_weight': 'balanced_subsample'}. Best is trial 1 with value: 0.71083613895946.
[I 2026-06-25 23:27:06,335] Trial 2 finished with value: 0.705640886258729 and parameters: {'n_estimators': 100, 'criterion': 'gini', 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': True, 'c

f1_macro_avg,▅▇▅▅▇▄▃▅▅▅▅█▅▇▇▆▅▆▅▆▂▇▆▆▇▇▃█▆▅▆▆▇▇▇▆▆▅▆▁
trial,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
best_cv_score,0.71498
f1_macro_avg,0.69051
test_f1_macro,0.70808
trial,49


In [16]:
study.best_params

{'n_estimators': 150,
 'criterion': 'gini',
 'max_depth': 11,
 'min_samples_split': 9,
 'min_samples_leaf': 3,
 'max_features': 'sqrt',
 'bootstrap': True,
 'class_weight': 'balanced_subsample'}

In [17]:
study.best_value

0.7149805440447522

In [19]:
save_result_path = Path("artifacts/best_params")

save_result_path.mkdir(parents=True, exist_ok=True)

result = {
    "model_name": "randomForest",
    "best_trial": study.best_trial.number,
    "best_params": study.best_params,
    "best_value": float(study.best_value)
}

with open(save_result_path / "random_forest.yaml", "w") as f:
    yaml.safe_dump(result, f, sort_keys=False)
